# F1AeroNet — VTP Image Generation

Generates separate **real (CFD ground truth)** and **predicted (GEM-CNN)** PNGs
for `N_VIZ` test samples.

**Outputs (written to `viz/`):**
- `{design_id}_real.png`      — ground-truth Cp + WSS (2×3 grid)
- `{design_id}_predicted.png` — GEM-CNN predictions   (2×3 grid)

Both PNGs for each sample share the same colour scale so they are directly comparable.

Run from the project root (`f1_aero_gem/`).

In [ ]:
# ── Cell 1 · Imports ─────────────────────────────────────────────────────────
import os
import sys
import json
import time
import numpy as np

sys.path.insert(0, os.path.abspath('.'))

import torch
import yaml
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

from models.f1_net import F1AeroNet
from data.drivaernet_dataset import DrivAerNetDataset, make_synthetic_dataset

print(f'[INFO] torch {torch.__version__}  |  numpy {np.__version__}')

In [ ]:
# ── Cell 2 · Config / Paths ───────────────────────────────────────────────────
CHECKPOINT  = 'new_final_run/best.pt'
CONFIG_YAML = 'configs/f1_base.yaml'
DATA_ROOT   = 'data/drivaernet_real'
OUT_DIR     = 'viz'
N_VIZ       = 3           # number of test samples to render

os.makedirs(OUT_DIR, exist_ok=True)

# Physical constants (SI)
RHO     = 1.225
U_INF   = 83.33
MU      = 1.81e-5
L_REF   = 4.6
TAU_REF = MU * U_INF / L_REF

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f'[INFO] Device  : {DEVICE}')
print(f'[INFO] OUT_DIR : {os.path.abspath(OUT_DIR)}')
print(f'[INFO] N_VIZ   : {N_VIZ}  (real + predicted PNG per sample)')

In [ ]:
# ── Cell 3 · Load Model ───────────────────────────────────────────────────────
if not os.path.exists(CHECKPOINT):
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT}')

print(f'[INFO] Loading: {CHECKPOINT}')
ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)

if 'cfg' in ckpt:
    saved_cfg = ckpt['cfg']
    cfg = saved_cfg.get('model', saved_cfg)
    print(f"[INFO] epoch {ckpt.get('epoch','?')}  best_val={ckpt.get('best_val', float('nan')):.6f}")
elif os.path.exists(CONFIG_YAML):
    with open(CONFIG_YAML) as fh:
        cfg = yaml.safe_load(fh).get('model', {})
    print(f'[WARNING] Falling back to {CONFIG_YAML}')
else:
    cfg = {}

model = F1AeroNet.from_config(cfg)
model.load_state_dict(ckpt['model'])
model = model.to(DEVICE)
model.eval()

pc = model.count_parameters()
print(f"[INFO] {pc['total']:,} params  (GEM: {pc['gem_blocks']:,}  cp_head: {pc['cp_head']:,}  cd_head: {pc['cd_head']:,})")

In [ ]:
# ── Cell 4 · Cd Normalisation Stats ──────────────────────────────────────────
cd_stats_path = os.path.join(DATA_ROOT, 'cd_stats.json')

if os.path.exists(cd_stats_path):
    with open(cd_stats_path) as fh:
        cd_stats = json.load(fh)
    CD_MEAN = float(cd_stats['cd_mean'])
    CD_STD  = float(cd_stats['cd_std'])
    print(f'[INFO] Cd stats: mean={CD_MEAN:.4f}  std={CD_STD:.4f}')
else:
    cache_dir = os.path.join(DATA_ROOT, 'processed', 'train')
    cds = []
    if os.path.isdir(cache_dir):
        for fn in os.listdir(cache_dir):
            if fn.endswith('.pt'):
                try:
                    d = torch.load(os.path.join(cache_dir, fn), weights_only=False)
                    if hasattr(d, 'y_cd') and d.y_cd is not None:
                        cds.append(float(d.y_cd.reshape(-1)[0]))
                except Exception:
                    pass
    if len(cds) >= 2:
        t = torch.tensor(cds, dtype=torch.float32)
        CD_MEAN, CD_STD = float(t.mean()), float(t.std())
        with open(cd_stats_path, 'w') as fh:
            json.dump({'cd_mean': CD_MEAN, 'cd_std': CD_STD}, fh)
        print(f'[INFO] Computed from {len(cds)} train samples: mean={CD_MEAN:.4f}  std={CD_STD:.4f}')
    else:
        CD_MEAN, CD_STD = 0.0, 1.0
        print('[WARNING] cd_stats.json not found — using mean=0, std=1.')

In [ ]:
# ── Cell 5 · Load Dataset ─────────────────────────────────────────────────────
USE_SYNTHETIC = False
dataset = None

if os.path.isdir(DATA_ROOT):
    for split_name in ('test', 'val'):
        try:
            ds = DrivAerNetDataset(DATA_ROOT, split=split_name)
            ds.set_cd_stats(CD_MEAN, CD_STD)
            if len(ds) > 0:
                dataset = ds
                SPLIT_NAME = split_name
                print(f"[INFO] '{split_name}' split: {len(ds)} samples")
                break
        except Exception as exc:
            print(f"[INFO] Could not load '{split_name}': {exc}")

if dataset is None or len(dataset) == 0:
    print('[WARNING] Real dataset unavailable — falling back to synthetic.')
    dataset = make_synthetic_dataset(n_meshes=12, n_vertices=500)
    SPLIT_NAME = 'synthetic'
    USE_SYNTHETIC = True

N_SAMPLES = len(dataset)
print(f'[INFO] {N_SAMPLES} samples total  |  rendering first {N_VIZ}')

In [ ]:
# ── Cell 6 · Denormalisation Utilities ───────────────────────────────────────
def symlog_inv(x):
    return np.sign(x) * (np.exp(np.abs(x)) - 1.0)

def denorm_cp(cp_norm, sample):
    sl_mean = float(sample.cp_sl_mean) if hasattr(sample, 'cp_sl_mean') else 0.0
    sl_std  = float(sample.cp_sl_std)  if hasattr(sample, 'cp_sl_std')  else 1.0
    return symlog_inv(cp_norm * sl_std + sl_mean)

def denorm_wss(wss_norm, sample):
    if hasattr(sample, 'wss_sl_mean') and sample.wss_sl_mean is not None:
        m = sample.wss_sl_mean.numpy().reshape(1, 3)
        s = sample.wss_sl_std.numpy().reshape(1, 3)
    else:
        m = np.zeros((1, 3), dtype=np.float32)
        s = np.ones((1, 3),  dtype=np.float32)
    return symlog_inv(wss_norm * s + m) * TAU_REF

def denorm_cd(cd_norm_val):
    return cd_norm_val * CD_STD + CD_MEAN

print(f'[INFO] TAU_REF = {TAU_REF:.4e} Pa')

In [ ]:
# ── Cell 7 · Inference Loop ────────────────────────────────────────────────────
def _sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    elif DEVICE.type == 'mps':
        torch.mps.synchronize()

# Warm-up
with torch.no_grad():
    s0 = dataset[0]
    _ = model(
        x=s0.x.to(DEVICE), edge_index=s0.edge_index.to(DEVICE),
        angles=s0.edge_angles.to(DEVICE), transporters=s0.edge_transporters.to(DEVICE),
        batch=torch.zeros(s0.num_nodes, dtype=torch.long, device=DEVICE)
    )
    _sync()
print('[INFO] Warm-up done. Running inference ...')

results = []
for idx in range(N_SAMPLES):
    sample = dataset[idx]
    did    = getattr(sample, 'design_id', f'sample_{idx:04d}')

    _sync()
    t0 = time.perf_counter()
    with torch.no_grad():
        pred = model(
            x=sample.x.to(DEVICE),
            edge_index=sample.edge_index.to(DEVICE),
            angles=sample.edge_angles.to(DEVICE),
            transporters=sample.edge_transporters.to(DEVICE),
            batch=torch.zeros(sample.num_nodes, dtype=torch.long, device=DEVICE)
        )
    _sync()
    elapsed_s = time.perf_counter() - t0

    cp_norm_pred  = pred['cp'].cpu().numpy()
    wss_norm_pred = pred['wss'].cpu().numpy()
    cd_norm_pred  = float(pred['cd'].cpu().item())

    cp_norm_true  = sample.y_cp.numpy()  if sample.y_cp  is not None else np.zeros_like(cp_norm_pred)
    wss_norm_true = sample.y_wss.numpy() if sample.y_wss is not None else np.zeros_like(wss_norm_pred)
    cd_norm_true  = float(sample.y_cd.reshape(-1)[0])

    vertices = sample.pos.numpy() if hasattr(sample, 'pos') else np.zeros((sample.num_nodes, 3))
    faces    = (sample.face.numpy().T.astype(np.int64)
                if (hasattr(sample, 'face') and sample.face is not None)
                else np.zeros((0, 3), dtype=np.int64))

    results.append({
        'design_id':     did,
        'elapsed_s':     elapsed_s,
        'cp_phys_pred':  denorm_cp(cp_norm_pred,  sample),
        'cp_phys_true':  denorm_cp(cp_norm_true,  sample),
        'wss_phys_pred': denorm_wss(wss_norm_pred, sample),
        'wss_phys_true': denorm_wss(wss_norm_true, sample),
        'cd_phys_pred':  denorm_cd(cd_norm_pred),
        'cd_phys_true':  denorm_cd(cd_norm_true),
        'vertices':      vertices,
        'faces':         faces,
        'n_vertices':    vertices.shape[0],
        'n_faces':       faces.shape[0],
    })

    print(f'  [{idx+1:3d}/{N_SAMPLES}]  {did:<28s}  '
          f'Cd_pred={denorm_cd(cd_norm_pred):.4f}  Cd_true={denorm_cd(cd_norm_true):.4f}  '
          f't={elapsed_s*1000:.1f}ms')

print(f'\n[INFO] Done. {N_SAMPLES} samples.')

In [ ]:
# ── Cell 8 · Real vs Predicted PNGs (N_VIZ samples) ─────────────────────────
#
# For each of the first N_VIZ results two PNGs are written:
#   {design_id}_real.png      — CFD ground truth
#   {design_id}_predicted.png — GEM-CNN prediction
#
# Each PNG is a 2×3 grid:
#   Row 0: Cp            (side / top / front views)
#   Row 1: WSS magnitude (side / top / front views)
#
# Shared colour scale across real + predicted for direct visual comparison.

def _panel(ax, verts, vals, faces, cmap, vmin, vmax, title, xdim=0, ydim=1):
    """Single projected view — tripcolor if faces exist, scatter fallback."""
    x, y   = verts[:, xdim], verts[:, ydim]
    artist = None
    if faces.shape[0] > 0:
        try:
            tri    = mtri.Triangulation(x, y, faces)
            artist = ax.tripcolor(tri, vals, cmap=cmap, vmin=vmin, vmax=vmax,
                                  shading='gouraud', rasterized=True)
        except Exception:
            pass
    if artist is None:
        artist = ax.scatter(x, y, c=vals, cmap=cmap, vmin=vmin, vmax=vmax,
                            s=1.0, rasterized=True)
    ax.set_title(title, fontsize=8, color='#ccc')
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='#aaa', labelsize=6)
    for sp in ax.spines.values():
        sp.set_color('#444')
    return artist


def render_sample(r, kind, out_dir):
    """
    Render one sample.
    kind='real'      -> ground-truth  Cp + WSS (CFD)
    kind='predicted' -> predicted     Cp + WSS (GEM-CNN)
    """
    did   = r['design_id']
    verts = r['vertices']
    faces = r['faces']

    if kind == 'real':
        cp_vals = r['cp_phys_true']
        wss_mag = np.linalg.norm(r['wss_phys_true'], axis=1)
        cd_val  = r['cd_phys_true']
        label   = 'Ground Truth (CFD)'
    else:
        cp_vals = r['cp_phys_pred']
        wss_mag = np.linalg.norm(r['wss_phys_pred'], axis=1)
        cd_val  = r['cd_phys_pred']
        label   = 'GEM-CNN Prediction'

    # Shared colour range
    cp_all  = np.concatenate([r['cp_phys_true'], r['cp_phys_pred']])
    wss_all = np.concatenate([
        np.linalg.norm(r['wss_phys_true'], axis=1),
        np.linalg.norm(r['wss_phys_pred'], axis=1),
    ])
    cp_vmin,  cp_vmax  = float(np.percentile(cp_all,  2)), float(np.percentile(cp_all,  98))
    wss_vmin, wss_vmax = float(np.percentile(wss_all, 2)), float(np.percentile(wss_all, 98))

    views = [
        (0, 2, 'Side (x-z)'),
        (0, 1, 'Top (x-y)'),
        (1, 2, 'Front (y-z)'),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), facecolor='#111111')
    fig.patch.set_facecolor('#111111')

    last_cp = last_wss = None
    for col, (xd, yd, view) in enumerate(views):
        last_cp  = _panel(axes[0, col], verts, cp_vals, faces,
                          'RdBu_r', cp_vmin, cp_vmax, f'Cp — {view}', xd, yd)
        last_wss = _panel(axes[1, col], verts, wss_mag, faces,
                          'hot',    wss_vmin, wss_vmax, f'WSS Mag (Pa) — {view}', xd, yd)

    if last_cp is not None:
        cb1 = fig.colorbar(last_cp,  ax=axes[0, :].tolist(), fraction=0.015, pad=0.02)
        cb1.set_label('Cp (dimensionless)', color='#ccc', fontsize=8)
        cb1.ax.yaxis.set_tick_params(color='#ccc')
        plt.setp(cb1.ax.yaxis.get_ticklabels(), color='#ccc', fontsize=7)

    if last_wss is not None:
        cb2 = fig.colorbar(last_wss, ax=axes[1, :].tolist(), fraction=0.015, pad=0.02)
        cb2.set_label('WSS Magnitude (Pa)', color='#ccc', fontsize=8)
        cb2.ax.yaxis.set_tick_params(color='#ccc')
        plt.setp(cb2.ax.yaxis.get_ticklabels(), color='#ccc', fontsize=7)

    fig.suptitle(f'{label} — {did}  |  Cd = {cd_val:.4f}',
                 color='white', fontsize=12, y=1.01)
    plt.tight_layout()

    out_path = os.path.join(out_dir, f'{did}_{kind}.png')
    fig.savefig(out_path, dpi=130, bbox_inches='tight', facecolor='#111111')
    plt.close(fig)
    return out_path


# ── Render ────────────────────────────────────────────────────────────────────
print(f'[INFO] Rendering {N_VIZ} sample(s) ...\n')

for r in results[:N_VIZ]:
    did = r['design_id']
    print(f'[RENDER] {did}  (V={r["n_vertices"]:,}, F={r["n_faces"]:,})')

    real_path = render_sample(r, 'real',      OUT_DIR)
    pred_path = render_sample(r, 'predicted', OUT_DIR)

    print(f'  real      → {os.path.basename(real_path)}')
    print(f'  predicted → {os.path.basename(pred_path)}')

print(f'\n[DONE] {N_VIZ * 2} PNGs saved to {os.path.abspath(OUT_DIR)}/')